### 梯度爆炸与梯度消失

#### 1. 核心原因

反向传播使用链式法则：

$$
\frac{\partial L}{\partial W} =
\prod_{i=1}^{n} \frac{\partial a_i}{\partial a_{i-1}}
$$

当多层梯度连乘时：

- 若每层梯度 < 1 → 梯度消失
- 若每层梯度 > 1 → 梯度爆炸

---
#### 2. 梯度消失
若每层梯度约为 0.5：
$$
0.5^n \to 0
$$
梯度迅速趋近于 0，前层几乎无法更新。
常见原因：
- sigmoid / tanh 激活函数
- 网络过深
- RNN 长序列

---
#### 3. 梯度爆炸
若每层梯度 > 1：

$$
2^n \to \infty
$$

梯度指数增长，导致：

- loss 震荡
- 参数溢出
- NaN

---

#### 4. 解决方法

梯度消失：
- ReLU
- Xavier / He 初始化
- Residual Connection
- LSTM / GRU
- BatchNorm / LayerNorm

梯度爆炸：
- Gradient Clipping （梯度剪裁，是一种直接处理）：torch.nn.utils.clip_grad_norm_（）通过梯度的范数来切割，torch.nn.utils.clip_grad_value_(parameters, clip_value)（直接切割值的范围）都在 loss.backward() 之后、optimizer.step() 之前调用。
- 合理初始化 Xavier kaiming_uniform_
- 正则化(相较于梯度clip是一种间接缓解，限制权重变大， 训练过程持续生效) L1 , L2（L2 -即Adam上加weight decay）

#### 5.项目中的经验
优先找自己代码中是否存在基本错误或者逻辑错误(数据异常[非常可能], loss 计算错误, backward 调用顺序错误
)，这是发生梯度异常的最大原因，没有后问题再去调试超参数（lr,）看问题能否被缓解，最后才是排查正则和进行梯度截断
因为我发现有时候就算加入了很多约束或者强制剪裁，也不会让模型训好，其实是代码中存在问题。
比如：
（1）数据或中间 Tensor 出现 NaN（最常见）
- 输入数据含 NaN / Inf
- 除以 0
- log(0)
- exp(x) 过大
- sqrt(负数)
（2）exp(100) → inf
softmax overflow
（3）叶子节点 (Leaf Tensor) 与梯度 - non-leaf tensor 梯度不会被保存，所以不能调用
什么是 leaf tensor
PyTorch 中：
直接创建的 tensor 是 leaf
计算得到的 tensor 是 non-leaf \\

'''
w = torch.randn(3, requires_grad=True)  # leaf\\
y = w * 2                                # non-leaf
y.grad = None
'''
| 类型              | 是否存梯度       |
| --------------- | ----------- |
| leaf tensor     | `grad` 会被保存 |
| non-leaf tensor | 默认不保存       |

#### 4. 总结
数据异常 > 学习率过大 > 数值运算不稳定 > 模型结构问题


# 激活函数

# 激活函数（Activation Function）

## 1. 为什么需要激活函数

如果神经网络只有 **线性变换**：

$$
y = W_2(W_1 x)
$$

多个线性层可以合并为：

$$
y = W x
$$

网络仍然是 **线性模型**，表达能力差。

因此需要 **非线性激活函数**，使网络能够拟合复杂函数。

---

## 2. 常见激活函数

### Sigmoid

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

特点：

- 输出范围：$(0,1)$，输出很平滑
- 常用于 **二分类输出层**， 因为输出相当于进行了一次归一化

问题：

- 梯度消失：值很大或者很小时梯度几乎为0
- 非零中心，输出恒等于正，更新梯度容易出现偏置偏移，导致权重朝一个方向更新
- 计算量也比较大

---

### Tanh

$$
tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

特点：
- 一种平滑（ReLU不平滑）、对称的激活函数，非常适合需要对负输入有敏感度
- 零中心， 激活值分布更加均衡
- 输出范围：$(-1,1)$
- 比 sigmoid 更常用（图像输入为-1 - 1时最后生成输出也需要-1 - 1 ， RNN中）

问题：

- 仍然存在梯度消失（值较大或较小时）

---

### ReLU

$$
ReLU(x) = \max(0,x)
$$

特点：

- 计算简单、成本低
- 缓解梯度消失
- 训练速度快

问题：
- 非零中心
- **Dead ReLU** 一个神经元输入一直为负数，梯度永远为0

$$
x < 0 \Rightarrow gradient = 0
$$

---

### Leaky ReLU

$$
f(x) =
\begin{cases}
x & x>0 \\
\alpha x & x \le 0
\end{cases}
$$

特点：

- 避免 Dead ReLU
- $\alpha$ 通常为 0.01

---

### GELU（Transformer常用）

$$
GELU(x) = x \Phi(x)
$$

近似形式：

$$
0.5x\left(1+\tanh\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right)
$$

特点：

- 更平滑
- Transformer / BERT 常用

---

## 3. 常见使用场景

|任务|激活函数|
|---|---|
|隐藏层（传统）|ReLU |
|Transformer / LLM|GELU |
|二分类输出|Sigmoid |
|多分类输出|Softmax |
|RNN早期|Tanh |

---

## 4. 面试常见问题

### 为什么 ReLU 比 sigmoid 好

原因：

1. sigmoid 梯度最大只有 0.25 → 容易梯度消失  
2. sigmoid 非零中心 → 收敛慢  
3. ReLU 计算更简单  

---

### 为什么 Transformer 使用 GELU

原因：

- GELU 是 **平滑 gating**
- 比 ReLU 更连续
- 在 NLP 任务上效果更好

---

## 5. 一句话总结

激活函数的作用是 **为神经网络引入非线性表达能力**，常见选择：

- CNN：ReLU  
- Transformer / LLM：GELU  
- 输出层：Sigmoid / Softmax


In [ ]:
# 激活函数实现
import numpy as np

def activation(fuc = 'sigmoid', x = None):
    if fuc == 'sigmoid':
        return 1/(1 + np.exp(-x))
    elif fuc == 'tanh':
        return(np.tanh(x))
    elif fuc == 'relu':
        return 0 if x <= 0 else x # np.maximum(0, x)

# 损失函数

## 1. 回归任务

### MSE（Mean Squared Error）均方差（会取均值） 、 L2范数(不是均方差，这是向量本身性质) 、欧式距离
含义有小差异L2衡量度量空间中向量的长度，均方差衡量两点差距
$$
L = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

特点：

- 对 **大误差更敏感**
- 最常见回归损失

PyTorch：
```
torch.nn.MSELoss()
```
### MAE （Mean Absolute Error） L1范数

$$
L = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

特点：

对异常值更鲁棒

梯度恒定

PyTorch：
```
torch.nn.L1Loss()
```
### Huber Loss
$$
L =
\begin{cases}
\frac{1}{2}(y-\hat{y})^2 & |y-\hat{y}|<\delta \
\delta(|y-\hat{y}|-\frac{1}{2}\delta) & |y-\hat{y}|\ge\delta
\end{cases}
$$

特点：

结合 MSE 与 MAE

对异常值更稳定

PyTorch：
```
torch.nn.SmoothL1Loss()
```

## 2. 分类任务

### BCE二分类

$$
L = -[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]
$$
PyTorch：

torch.nn.BCELoss()

torch.nn.BCEWithLogitsLoss()

### Cross Entropy Loss多分类

$$
L = -\sum y_i \log(\hat{y}_i)
$$

torch.nn.CrossEntropyLoss()
特点：

内部包含 softmax
分类任务最常用

分类损失的本质内涵：BCE / CE 的本质是极大似然估计（MLE），而经验风险是用样本均值近似期望（类似 Monte Carlo 估计）。

## 3. 多模态
自监督学习， 对比损失
$$
L =
-\log
\frac{e^{sim(x_i,y_i)/\tau}}
{\sum_j e^{sim(x_i,y_j)/\tau}}
$$
sim(x_i,y_i) 是正样本对的

作用：
拉近正样本

推远负样本

# 过拟合（Overfitting）与欠拟合（Underfitting）
## 欠拟合（Underfitting）

模型 过于简单，无法学习数据中的真实规律。

#### 表现：

训练误差高

测试误差高

#### 常见原因：

模型容量太小

训练不足（epoch 少）

特征不足

正则化过强

#### 解决欠拟合

增加模型复杂度（更深网络）

增加训练时间

减少正则化

增加特征

## 过拟合（Overfitting）

模型 过于复杂，记住了训练数据中的噪声。

#### 表现：

训练误差低

测试误差高


#### 常见原因：

模型过大

数据量太小

训练过久

数据噪声多，导致模型过分记住噪声特征

#### 解决过拟合

增加数据量

数据增强（data augmentation）

正则化（L2 / weight decay）

Dropout

Early stopping

减小模型规模

# 优化器

优化器用于 **根据梯度更新模型参数**：

$$
\theta = \theta - \eta \nabla_\theta L
$$

其中：

* $\theta$：参数
* $\eta$：学习率
* $\nabla_\theta L$：梯度

---

# 1. SGD（Stochastic Gradient Descent）

更新：

$$
\theta = \theta - \eta g
$$

特点：

* 实现简单
* 收敛稳定
* 需要较好的 learning rate

PyTorch：

```python
torch.optim.SGD(model.parameters(), lr=0.01)
```

---

# 2. Momentum
原理来自于：EWA指数加权平均，动量通过指数加权平均的方式来计算。

加入 **动量**，减少震荡：

$$
v_t = \beta v_{t-1} + g_t
$$

$$
\theta = \theta - \eta v_t
$$

作用：

* 加速收敛
* 减少震荡
* 跳出局部最小值

PyTorch：

```python
torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
```

---

# 3. AdaGrad

根据历史梯度 **自适应调整学习率**：

$$
G_t = \sum g_t^2
$$

$$
\theta = \theta - \frac{\eta}{\sqrt{G_t+\epsilon}} g_t
$$

特点：

* 适合 **稀疏特征**

问题：

* 学习率会不断减小

PyTorch：

```python
torch.optim.Adagrad(model.parameters(), lr=0.01)
```

---

# 4. RMSProp

解决 AdaGrad 学习率过小问题：

$$
E[g^2]*t = \beta E[g^2]*{t-1} + (1-\beta)g_t^2
$$

$$
\theta = \theta - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} g_t
$$

特点：

* 训练更稳定
* 常用于 RNN

PyTorch：

```python
torch.optim.RMSprop(model.parameters(), lr=0.001)
```

---

# 5. Adam（最常用）

结合：

* Momentum（1阶矩）
* RMSProp（2阶矩）

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t
$$

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$

更新：

$$
\theta = \theta - \eta \frac{m_t}{\sqrt{v_t}+\epsilon}
$$

特点：

* 自适应学习率
* 收敛快
* 大多数任务默认选择

PyTorch：

```python
torch.optim.Adam(model.parameters(), lr=1e-3)
```

---

# 6. AdamW（Transformer常用）

Adam + **正确的 weight decay**

区别：

Adam 中：

$$
L = L + \lambda ||W||^2
$$

AdamW 中：

$$
\theta = \theta - \eta \lambda \theta
$$

优点：

* 正则化效果更好
* Transformer / LLM 常用

PyTorch：

```python
torch.optim.AdamW(model.parameters(), lr=1e-4)
```

---

# 7. 面试速记

| 优化器      | 特点             |
| -------- | -------------- |
| SGD      | 最基础            |
| Momentum | 减少震荡           |
| AdaGrad  | 适合稀疏特征         |
| RMSProp  | 改进 AdaGrad     |
| Adam     | 最常用            |
| AdamW    | Transformer 常用 |
